# Current Results Summary

## Simple English First
This notebook summarizes results that are already present in committed reports or tracked artifacts. It does not invent new experiments.

If something is missing, the notebook marks it clearly as pending.

## What Counts As A Real Result Here
- committed markdown reports
- tracked CSV summaries
- tracked evaluation JSON files in `artifacts/`

Pending items are not treated as results.

In [1]:
from pathlib import Path
import json
import pandas as pd

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src').exists():
            return candidate
    raise FileNotFoundError('Could not locate the TyreVisionX repo root from the current working directory.')

repo_root = find_repo_root()
print('Repo root:', repo_root)
day3_paths = sorted((repo_root / 'artifacts/day3_baseline').glob('*/eval_report.json'))
rows = []
for path in day3_paths:
    report = json.loads(path.read_text(encoding='utf-8'))
    metrics = report.get('metrics', {})
    rows.append({
        'run': path.parent.name,
        'split': report.get('split'),
        'accuracy': metrics.get('accuracy'),
        'f1_defect': metrics.get('f1_defect'),
        'recall_defect': metrics.get('recall_defect'),
        'auroc': metrics.get('auroc'),
    })

day3_df = pd.DataFrame(rows).sort_values('run') if rows else pd.DataFrame()
day3_df

Repo root: /Users/ritik/Documents/Project TDA/TyreVisionX


,run,split,accuracy,f1_defect,recall_defect,auroc
0,frozen_resnet18_unfreeze_v2,test,0.964706,0.966038,0.984615,0.993292
1,frozen_resnet18_v1,test,0.933333,0.935361,NaN,0.980185
2,frozen_resnet50_unfreeze_v1,test,0.952941,0.955556,0.992308,0.996062
3,frozen_resnet50_unfreeze_v2_3ep,test,0.980392,0.980545,0.969231,0.998523
4,simple_cnn_v1,test,0.764706,0.810127,NaN,0.863077
5,simple_cnn_v2,test,0.835294,0.855172,NaN,0.889292


## Historical Baseline Interpretation

The strongest tracked results in this repository are still historical baseline results, especially from the Day 3 and Day 5 experiments.

### Technical detail
These results are meaningful for research discussion, but they should not be confused with fresh outputs from the cleaned canonical pipeline after the repository refactor.

In [2]:
summary_by_model = repo_root / 'artifacts/day5/longrun_seed_sweep/summary_by_model.csv'
if summary_by_model.exists():
    day5_df = pd.read_csv(summary_by_model)
    display(day5_df)
else:
    print('Pending: day5 summary_by_model.csv not available.')

,model_key,model,recall_mean,recall_std,precision_mean,precision_std,fn_mean,fn_std,fp_mean,fp_std,auprc_mean,auprc_std
0,baseline,Baseline,0.943590,0.084382,0.745787,0.053339,7.333333,10.969655,43.000000,14.730920,0.901674,0.011379
1,augmentation,+ Augmentation,0.941026,0.029123,0.778739,0.030068,7.666667,3.785939,35.000000,7.000000,0.946823,0.007912
2,dropout,+ Dropout,0.894872,0.108876,0.737010,0.072134,13.666667,14.153916,43.666667,18.929694,0.887496,0.015746
3,batchnorm,+ BatchNorm,0.864103,0.103870,0.765983,0.130173,17.666667,13.503086,39.000000,28.513155,0.887344,0.047774


In [3]:
best_day3 = day3_df.sort_values('f1_defect', ascending=False).head(5) if not day3_df.empty else pd.DataFrame()
best_day3

,run,split,accuracy,f1_defect,recall_defect,auroc
3,frozen_resnet50_unfreeze_v2_3ep,test,0.980392,0.980545,0.969231,0.998523
0,frozen_resnet18_unfreeze_v2,test,0.964706,0.966038,0.984615,0.993292
2,frozen_resnet50_unfreeze_v1,test,0.952941,0.955556,0.992308,0.996062
1,frozen_resnet18_v1,test,0.933333,0.935361,NaN,0.980185
5,simple_cnn_v2,test,0.835294,0.855172,NaN,0.889292


## Canonical Pipeline Result Status
Pending:
- fresh `artifacts/experiments/...` checkpoint from the cleaned canonical path
- canonical-pipeline report JSON for D1
- any tracked D2/D3 canonical evaluation report

These are intentionally left as pending rather than filled with guessed values.

## Suggested Talking Points
- The repo has useful historical baselines.
- The cleaned canonical path is structurally clearer than before.
- The next important step is to run and record a fresh canonical D1 experiment.
- Cross-dataset discussion should remain tentative until D2 and D3 are populated.

## Future Work Sections
- Anomaly detection: pending, no tracked experiments yet.
- Localization: pending, no tracked spatial artifacts yet.
- Segmentation: pending, currently stub-level only.
- Multi-view 3D: pending, roadmap only.
- Knowledge reasoning: pending, roadmap only.